# Classic Convolutional Neural Network (CNN) Architectures — Instructor Edition


- Name: [Your Name]

- Student ID: [Your Student ID]

## Experiment Objectives

In this experiment, you will use **prompt-based programming** to build and train 4 classic CNN architectures (AlexNet, VGG, GoogleNet, ResNet) using the PyTorch framework on the CIFAR10 dataset. Through this experiment, you will:

1. **Understand the design philosophy** of classic CNN architectures: the evolution from AlexNet to ResNet
2. **Master core CNN components**: convolutional layers, pooling layers, batch normalization, residual connections, and Inception modules
3. **Implement a complete training and evaluation pipeline**: data loading, model definition, training, evaluation, and visualization
4. **Compare performance across architectures**: accuracy, parameter count, training time, and other metrics
5. **Understand the core principle of residual learning**: why the network still works even after removing several layers

### Instructions

Each code cell is preceded by a **Prompt**. Enter the prompt into your AI programming tool, then paste the generated code into the corresponding cell and run it. The instructor version contains the complete code implementation.


## 1. 1. Environment Setup


**Prompt**

Please help me set up the Python environment with the following requirements:

1. **Import the necessary libraries:**
   - `matplotlib.pyplot` and `matplotlib`
   - Configure matplotlib for general use:
   - `font.sans-serif` set to a list of standard fonts
   - `axes.unicode_minus` set to `False`


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

# Configure matplotlib settings
# Use default font (English notebook)
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']
plt.rcParams['axes.unicode_minus'] = False  # Fix negative sign display


## 2. 2. PyTorch Imports and CIFAR10 Dataset Loading


**Prompt**

Use the PyTorch framework to import necessary libraries and load the CIFAR10 dataset with the following requirements:

**1. Import necessary libraries:**
- `torch`, `torch.nn`, `torchvision.transforms`, `torch.utils.data.DataLoader`
- `numpy`, `matplotlib.pyplot`, `time`, `os`
- - `confusion_matrix`, `classification_report`, `accuracy_score` from `sklearn.metrics`
- `seaborn`

**2. Set global parameters:**
- Data path: `data_path = './data'`
- Batch size: `batch_size = 64` (reduce to 32 if GPU memory is limited)
- Number of epochs: `num_epochs = 10`

**3. Define data preprocessing:**
- Training transforms: `RandomHorizontalFlip()` → `ToTensor()` → `Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])`
- Test transforms: `ToTensor()` → `Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])`

**4. Load datasets:**
- Training set: `torchvision.datasets.CIFAR10(root=data_path, train=True, transform=train_transform, download=True)`
- Test set: `torchvision.datasets.CIFAR10(root=data_path, train=False, transform=test_transform, download=True)`
- Create data loaders with `DataLoader`, set `num_workers=2`

**5. Define class names:**
- `classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')`

**6. Implement image display function `imshow(img)`:
- Denormalize, convert to numpy, transpose to (H,W,C), display with plt.imshow


In [ ]:
# Import necessary libraries (PyTorch version)
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import time
import os
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns

# Set dataset download path
data_path = './data'
batch_size = 64
num_epochs = 10

# Define data preprocessing (training: with augmentation; test: normalization only)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Download and load training set
trainset = torchvision.datasets.CIFAR10(
    root=data_path,
    train=True,
    transform=train_transform,
    download=True)

trainloader = DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2)

# Download and load test set
testset = torchvision.datasets.CIFAR10(
    root=data_path,
    train=False,
    transform=test_transform,
    download=True)

testloader = DataLoader(
    testset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2)

# CIFAR10 class names
classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# Image display function
def imshow(img):
    img = img / 2 + 0.5  # Denormalize
    npimg = img.cpu().numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis('off')
    plt.show()

# View sample images
for batch_id, data in enumerate(trainloader):
    images, labels = data
    break

fig, axes = plt.subplots(2, 2)
for i in range(4):
    row, col = i // 2, i % 2
    img = images[i]
    img = img / 2 + 0.5
    npimg = img.cpu().numpy()
    npimg = np.transpose(npimg, (1, 2, 0))
    axes[row, col].imshow(npimg)
    axes[row, col].set_title(classes[labels[i].item()])
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

# Print dataset info
print(f'Training set size: {len(trainset)}')
print(f'Test set size: {len(testset)}')
print(f'Image shape: {trainset[0][0].shape}')


## 3. 3. Utility Functions

### 3.1 3.1 Random Seed and Device Setup


**Prompt**

Set random seeds and check the computing device with the following requirements:
1. Use `torch.manual_seed(42)` and `np.random.seed(42)` to set random seeds
2. Get the current computing device using `torch.device('cuda' if torch.cuda.is_available() else 'cpu')` and print it


In [ ]:
# TODO: add your code here

### 3.2 3.2 General Training Function


**Prompt**

Create a general-purpose model training function with the following requirements:

**1. Function signature:** `train_model(model, trainloader, epochs=10, lr=0.001)`
- Parameters: model instance, training data loader, number of epochs, learning rate
- Returns: `(train_losses, train_accs, trained_model)`

**2. Inside the function:**
- Define loss function: `nn.CrossEntropyLoss()`
- Define optimizer: `torch.optim.Adam(model.parameters(), lr=lr)`
- Create empty lists `train_losses` and `train_accs`

**3. Training loop (per epoch):**
- `model.train()`
- Iterate over `trainloader`, get `images, labels`
- Forward pass → compute loss → backward pass → update parameters → zero gradients
- Accumulate total loss, correct predictions, and total samples
- Compute average loss and accuracy for the epoch, append to lists, and print

**4. After training:**
- Plot training loss and accuracy curves (1 row, 2 columns)
- Return the recorded data and trained model


In [ ]:
# TODO: add your code here.

### 3.3 3.3 General Evaluation Function


**Prompt**

Create a general-purpose model evaluation function with the following requirements:

**1. Function signature:** `evaluate_model(model, testloader, class_names=None)`
- Parameters: model instance, test data loader, list of class names
- Goal: Evaluate model performance on the test set, output confusion matrix and classification report

**2. Inside the function:**
- `model.eval()`, use `torch.no_grad()` to disable gradient computation
- Iterate over `testloader`, collect all predictions and ground-truth labels
- Compute and plot confusion matrix heatmap (using seaborn.heatmap)
- Print classification report (precision, recall, f1-score)
- Compute and print overall accuracy and per-class accuracy


In [ ]:
# TODO: add your code here.

## 4. AlexNet

### 4.1 AlexNet Model Definition


**Prompt**

Implement the AlexNet architecture using PyTorch. Note: The original AlexNet was designed for 224×224 images; you need to adapt it for CIFAR10's 32×32 images.

**AlexNet Background:**
- Winner of the 2012 ImageNet competition, marking the beginning of the deep learning renaissance
- Key contributions: ReLU activation, GPU parallel training, Dropout regularization, data augmentation
- Architecture: 5 convolutional layers + 3 fully connected layers
- Input mean normalization: subtract mean from input images
- After the 5th convolutional layer, feature map size is 13×13×256; after 3×3 max pooling, it becomes 6×6×256; the FC layer input is 9216

**Requirements:**

**1. Class definition:** `AlexNet`, inheriting from `nn.Module`

**2. Network structure (CIFAR10-adapted):**
- `conv1`: Conv2D(3, 64, 5, padding=2) → BN → ReLU → MaxPool(3, stride=2, padding=1)
- `conv2`: Conv2D(64, 192, 5, padding=2) → BN → ReLU → MaxPool(3, stride=2, padding=1)
- `conv3`: Conv2D(192, 384, 3, padding=1) → BN → ReLU
- `conv4`: Conv2D(384, 256, 3, padding=1) → BN → ReLU
- `conv5`: Conv2D(256, 256, 3, padding=1) → BN → ReLU → MaxPool(3, stride=2, padding=1)
- `fc1`: Linear(4096, 1024) → ReLU → Dropout(0.5)
- `fc2`: Linear(1024, 1024) → ReLU → Dropout(0.5)
- `fc3`: Linear(1024, 10) (output layer, no activation)

**3. Conv block (optional):** You may define a `ConvBlock` class containing Conv+BN+ReLU+MaxPool to reduce repetitive code

**4. Forward pass `forward(self, x)`:
- Pass through convolutional layers, flatten, then fully connected layers
- Return output logits

**5. Weight initialization function:**
- Use Kaiming Normal initialization for Conv2d layers, initialize biases to 0
- Use Kaiming Normal initialization for Linear layers, initialize biases to 0


In [ ]:
# TODO: add your code here.

### 4.2 4.2 Training and Evaluating AlexNet


**Prompt**

Instantiate the AlexNet model and perform training and evaluation. Requirements:

1. **Instantiate the model:** `model = AlexNet(num_classes=10)`, print the model structure
2. **Train the model:** Call `train_model(model, trainloader, epochs=num_epochs, lr=0.001)`
3. **Evaluate the model:** Call `evaluate_model(model, testloader, class_names=classes)`


In [ ]:
# TODO: add your code here.

**Discussion Questions (AlexNet):**

1. **Input mean normalization:** AlexNet performs mean normalization on input images. How is this implemented in the CIFAR10 dataset? What is the role of `mean=[0.5,0.5,0.5]` in the Normalize operation?

2. **Feature map size changes:** In the original AlexNet, after the 5th conv layer, the feature map is 13×13×256; after 3×3 max pooling it becomes 6×6×256; the FC layer input is 9216=6×6×256. In your CIFAR10-adapted version, how do the feature map sizes change? Can you calculate the final flattened dimension?

3. **The role of Dropout:** AlexNet was the first to introduce Dropout. What role do you think it plays in preventing overfitting? When should Dropout be used during training?

4. **CIFAR10 accuracy:** What is the approximate accuracy of AlexNet on CIFAR10? How does it compare to the deeper networks that follow?


## 5. VGG

### 5.1 VGG Model Definition


**Prompt**

Implement the VGG network using PyTorch. VGG's core idea is to use successive small convolutional kernels (3×3) instead of large ones, stacking multiple 3×3 convolutions to increase network depth and receptive field.

**VGG Background:**
- Runner-up of the 2014 ImageNet competition, but with excellent transfer learning capabilities
- Uses only 3×3 kernels with stride 1 and padding 1
- 5 groups of convolutional layers + 3 fully connected layers
- The first two groups use fewer filters to capture primitive features; the last three groups use more filters to capture high-level semantic features
- Feature extraction can be viewed as convolving the image with 512 super filters, producing 512 feature maps for the fully connected layers
- Uses 2×2 max pooling (non-overlapping pooling) with stride 2
- VGG16: 13 conv layers + 3 FC layers; VGG11: 8 conv layers + 3 FC layers

**Requirements:**

**1. Class definition:** `VGG`, inheriting from `nn.Module`

**2. VGG configuration:** Use a VGG11-like configuration (simplified for CIFAR10):
- `conv1-2`: [64, 64] → MaxPool(2, stride=2)  → 16×16
- `conv3-4`: [128, 128] → MaxPool(2, stride=2) → 8×8
- `conv5-7`: [256, 256, 256] → MaxPool(2, stride=2) → 4×4
- `conv8-10`: [512, 512, 512] → MaxPool(2, stride=2) → 2×2
- `conv11-13`: [512, 512, 512] → MaxPool(2, stride=2) → 1×1
- FC1: 512 → 4096 → ReLU → Dropout(0.5)
- FC2: 4096 → 4096 → ReLU → Dropout(0.5)
- FC3: 4096 → num_classes

**Note:** For CIFAR10, after the final pooling, the feature map size is 1×1 (if using 5 pooling groups), so you can use AdaptiveAvgPool2d(1) to handle this uniformly

**3. Simplified implementation:** Define a `make_layers` function that generates convolutional layer sequences from a configuration list
- `cfg` config: e.g. `[64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M']`
- Numbers represent output channels for conv layers, 'M' represents max pooling

**4. Forward pass:**
- Through conv layers → AdaptiveAvgPool2d(1) → flatten → fully connected layers → output


In [ ]:
# TODO: add your code here.

### 5.2 5.2 Training and Evaluating VGG


**Prompt**

Instantiate the VGG model and perform training and evaluation. Requirements:

1. **Instantiate the model:** `model = VGG(cfg_key='VGG11', num_classes=10)`, print the model structure
2. **Train the model:** Call `train_model(model, trainloader, epochs=num_epochs, lr=0.001)`
3. **Evaluate the model:** Call `evaluate_model(model, testloader, class_names=classes)`


In [ ]:
# TODO: add your code here.

**Discussion Questions (VGG):**

1. **Advantages of small kernels:** VGG uses only 3×3 kernels. Two stacked 3×3 convolutions have a receptive field equivalent to one 5×5 convolution; three 3×3 are equivalent to a 7×7. What are the advantages of this approach? (Hint: Consider parameter count and number of nonlinear transformation layers.)

2. **Filter count pattern:** The first two groups use fewer filters (64, 128), while the last three use more (256, 512). What is the principle behind this design? Why use fewer filters in shallow layers and more in deep layers?

3. **512 feature maps:** VGG's feature extraction can be viewed as using 512 super filters to produce 512 feature maps for the FC layers. What information do these 512 feature maps encode about the image?

4. **Non-overlapping pooling:** VGG uses 2×2 pooling kernels with stride 2 (non-overlapping pooling). How does this differ from AlexNet's 3×3 pooling with stride 2 (overlapping pooling)? What are the advantages and disadvantages of each?

5. **Parameter count comparison:** Compared to AlexNet, does VGG have more or fewer parameters? What problems do you think VGG's high parameter count might cause?


## 6. GoogleNet (Inception)

### 6.1 Inception Module Definition


**Prompt**

GoogleNet (also known as Inception-v1) was the winner of the 2014 ImageNet competition. Its core innovation is the Inception module — using parallel convolutional kernels of different scales in the same layer to extract features, then concatenating them.

**GoogleNet Background:**
- 2014 ImageNet winner, surpassing VGG with 22 layers of depth
- Core innovation: Inception module (multi-scale parallel convolutions)
- Classification layer outputs a 1024-dimensional vector
- Uses 1×1 convolutions for dimensionality reduction to control computational cost
- Auxiliary classifiers add extra supervision signals at intermediate layers
- Global Average Pooling replaces some fully connected layers

**Inception Module Structure:**
- Branch 1: 1×1 convolution
- Branch 2: 1×1 conv + 3×3 conv
- Branch 3: 1×1 conv + 5×5 conv
- Branch 4: 3×3 max pooling + 1×1 conv
- Concatenate outputs of all 4 branches along the channel dimension

**Requirements:**

**1. Class definition:** `Inception`, inheriting from `nn.Module`

**2. Constructor `__init__(self, in_channels, ch1x1, ch3x3_reduce, ch3x3, ch5x5_reduce, ch5x5, pool_proj)`:
- `branch1`: 1×1 conv (output `ch1x1` channels)
- `branch2`: 1×1 conv (output `ch3x3_reduce`) → 3×3 conv (padding=1, output `ch3x3`)
- `branch3`: 1×1 conv (output `ch5x5_reduce`) → 5×5 conv (padding=2, output `ch5x5`)
- `branch4`: 3×3 max pooling (padding=1, stride=1) → 1×1 conv (output `pool_proj`)

**3. Forward pass `forward(self, x)`:
- Compute outputs of all 4 branches
- Concatenate using `torch.cat([branch1, branch2, branch3, branch4], dim=1)`
- Return the concatenated result


In [ ]:
# TODO: add your code here.

### 6.2 GoogleNet Model Definition


**Prompt**

Build a complete GoogleNet network using Inception modules, adapted for the CIFAR10 dataset.

**GoogleNet Architecture (CIFAR10-adapted):**

1. **Stem (initial convolutional layers):**
   - Conv(3→64, 3×3, stride=1, padding=1) → BN → ReLU → MaxPool(3×3, stride=2, padding=1)
   - Conv(64→192, 3×3, stride=1, padding=1) → BN → ReLU → MaxPool(3×3, stride=2, padding=1)

2. **Inception module sequence:**
   - inception(192, 64, 96, 128, 16, 32, 32) → output 256 channels
   - inception(256, 128, 128, 192, 32, 96, 64) → output 480 channels
   - MaxPool(3×3, stride=2, padding=1)
   - inception(480, 192, 96, 208, 16, 48, 64) → output 512 channels
   - inception(512, 160, 112, 224, 24, 64, 64) → output 528 channels
   - inception(528, 128, 128, 256, 24, 64, 64) → output 576 channels
   - inception(576, 112, 144, 288, 32, 64, 64) → output 640 channels
   - inception(640, 256, 160, 320, 32, 128, 128) → output 768 channels
   - MaxPool(3×3, stride=2, padding=1)

3. **Classifier:**
   - AdaptiveAvgPool2d(1) → flatten
   - Dropout(0.4)
   - Linear(768, 1024) → ReLU (outputs 1024-dim vector, following the specification)
   - Linear(1024, num_classes)

**Requirements:** Define the `GoogleNet` class, inheriting from `nn.Module`, implementing the structure above. Initialize weights using Kaiming Normal.


In [ ]:
# TODO: add your code here.

### 6.3 6.3 Training and Evaluating GoogleNet


**Prompt**

Instantiate the GoogleNet model and perform training and evaluation. Requirements:

1. **Instantiate the model:** `model = GoogleNet(num_classes=10)`, print the model structure
2. **Train the model:** Call `train_model(model, trainloader, epochs=num_epochs, lr=0.001)`
3. **Evaluate the model:** Call `evaluate_model(model, testloader, class_names=classes)`


In [ ]:
# TODO: add your code here.

**Discussion Questions (GoogleNet):**

1. **Multi-scale features:** The Inception module uses 1×1, 3×3, and 5×5 convolutional kernels at the same layer. What types of features does each kernel size capture? Why are different kernel scales needed?

2. **The role of 1×1 convolutions:** The Inception module makes extensive use of 1×1 convolutions (especially at the beginning of each branch). Analyze the two core functions of 1×1 convolutions in the Inception module.

3. **1024-dim classification vector:** GoogleNet's classification layer outputs a 1024-dimensional vector (in this experiment, Linear(768, 1024)). Why 1024 dimensions? What information does this vector encode?

4. **Computational efficiency:** Compared to VGG, how does GoogleNet compare in terms of parameter count and computation? Why might a deeper network (22 layers) be more efficient than VGG (16-19 layers)?

5. **Auxiliary classifiers:** The original GoogleNet added auxiliary classifiers at intermediate layers. Analyze their role — how do they propagate gradients? Why do they help train deep networks?


## 7. ResNet

### 7.1 Residual Block Definition


**Prompt**

Implement the core component of ResNet — the Residual Block. Residual learning was the key innovation of the 2015 ImageNet winner.

**ResNet Background:**
- Winner of the 2015 ImageNet competition, with residual learning as its core innovation
- Introduced skip connections (shortcut connections)
- Solved the degradation problem of deep networks
- Even after removing several layers, the residual network still works properly
- Core idea: Let the network learn a residual mapping F(x) = H(x) - x, instead of learning H(x) directly

**Requirements:**

**1. Class definition:** `ResidualBlock`, inheriting from `nn.Module`

**2. Constructor `__init__(self, in_channels, out_channels, stride=1)`:
- `conv1`: Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False) → BN → ReLU
- `conv2`: Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False) → BN
- `shortcut`: If stride≠1 or in_channels≠out_channels, use 1×1 conv + BN for dimension matching; otherwise use `nn.Sequential()` (identity mapping)

**3. Forward pass `forward(self, x)`:
- Save input as `identity`
- `out = conv1(x)` → ReLU → `out = conv2(out)`
- `out = out + shortcut(identity)` (skip connection)
- `out = ReLU(out)`
- Return the result


In [ ]:
# TODO: add your code here.

### 7.2 ResNet Model Definition


**Prompt**

Build a complete ResNet network (simplified ResNet-18) using residual blocks, adapted for the CIFAR10 dataset.

**ResNet-18 Architecture (CIFAR10-adapted):**

1. **Initial convolution:**
   - Conv2d(3, 64, 3×3, stride=1, padding=1, bias=False) → BN → ReLU
   - Note: CIFAR10 is 32×32, no need for a 7×7 large kernel

2. **4 residual layers:**
   - `layer1`: 2 × ResidualBlock(64, 64, stride=1) → output 32×32×64
   - `layer2`: 2 × ResidualBlock(64, 128, stride=2) → output 16×16×128
   - `layer3`: 2 × ResidualBlock(128, 256, stride=2) → output 8×8×256
   - `layer4`: 2 × ResidualBlock(256, 512, stride=2) → output 4×4×512

3. **Classifier:**
   - AdaptiveAvgPool2d(1) → flatten
   - Linear(512, num_classes)

**Requirements:**

1. Define the `ResNet` class, inheriting from `nn.Module`
2. Implement the `_make_layer(self, in_channels, out_channels, num_blocks, stride)` helper function
3. Use Kaiming Normal weight initialization
4. Note the degradation problem — discuss why the network still works after removing several layers


In [ ]:
# TODO: add your code here.

### 7.3 7.3 Training and Evaluating ResNet


**Prompt**

Instantiate the ResNet model and perform training and evaluation. Requirements:

1. **Instantiate the model:** `model = ResNet(num_classes=10)`, print the model structure
2. **Train the model:** Call `train_model(model, trainloader, epochs=num_epochs, lr=0.001)`
3. **Evaluate the model:** Call `evaluate_model(model, testloader, class_names=classes)`


In [ ]:
# TODO: add your code here.

**Discussion Questions (ResNet):**

1. **Degradation problem:** He et al. noted in their paper that as network depth increases, accuracy saturates and then degrades, and this is not caused by overfitting (training error also increases). Why does the degradation problem occur? How do residual connections solve it?

2. **Removing layers still works:** A unique property of ResNet — even after removing several layers, the network still works. Explain this phenomenon mathematically. Hint: If a layer learns parameters close to zero (F(x) ≈ 0), the layer approximates an identity mapping.

3. **Gradient propagation through skip connections:** Analyze the advantage of skip connections from a backpropagation perspective. Why can residual structures train very deep networks (e.g., ResNet-152)?

4. **Identity mapping vs. projection mapping:** When input and output dimensions don't match, there are two shortcut options: (a) use 1×1 convolution for projection; (b) zero-pad the feature maps. What are the pros and cons of each?

5. **Residual networks and intuition:** The idea of residual learning can be understood from cognitive science — learning increments is easier than learning absolute values. Can you give similar examples from everyday life?


## 8. 8. Model Comparison Analysis


**Prompt**

Perform a multi-dimensional comparison analysis of the 4 classic CNN architectures. Requirements:

**1. Accuracy comparison bar chart:**
- Create a bar chart comparing test accuracies of all 4 models
- Title: 'Accuracy Comparison of Different CNN Architectures on CIFAR10'
- X-axis label: Model names (AlexNet, VGG, GoogleNet, ResNet)
- Y-axis label: 'Accuracy'
- Annotate exact accuracy values on each bar

**2. Training process comparison (subplots):**
- 1 row, 2 columns: left shows training loss curves, right shows training accuracy curves
- Use different colors for the 4 models (red, blue, green, orange)
- Add a legend
- Title: 'Training Process Comparison'

**3. Report parameter counts for each model (using manual calculation):**
- Compute and print the parameter count for each model (in millions)
- Print format: f'{model_name}: {params_m:.2f}M params'

**4. Training time comparison:**
- Show training time for each model using a bar chart

**5. Comprehensive comparison table:** Display as a text table:
| Model | Accuracy | Params | Training Time | Depth |
|------|----------|--------|---------------|-------|
| AlexNet | xx% | xxM | xxs | 8 layers |
| VGG | xx% | xxM | xxs | 11 layers |
| GoogleNet | xx% | xxM | xxs | ~14 layers |
| ResNet | xx% | xxM | xxs | 18 layers |

**Note:** If all 4 networks have been trained, use the previously saved variables for comparison.


In [ ]:
# Model comparison analysis

# Collect data
models_data = {
    'AlexNet': {
        'acc': alexnet_acc,
        'losses': alexnet_losses,
        'accs': alexnet_accs,
        'time': alexnet_time,
        'color': 'red'
    },
    'VGG11': {
        'acc': vgg_acc,
        'losses': vgg_losses,
        'accs': vgg_accs,
        'time': vgg_time,
        'color': 'blue'
    },
    'GoogleNet': {
        'acc': googlenet_acc,
        'losses': googlenet_losses,
        'accs': googlenet_accs,
        'time': googlenet_time,
        'color': 'green'
    },
    'ResNet18': {
        'acc': resnet_acc,
        'losses': resnet_losses,
        'accs': resnet_accs,
        'time': resnet_time,
        'color': 'orange'
    }
}

# 1. Accuracy comparison bar chart
plt.figure(figsize=(10, 6))
model_names = list(models_data.keys())
accuracies = [models_data[name]['acc'] for name in model_names]
colors = [models_data[name]['color'] for name in model_names]
bars = plt.bar(model_names, accuracies, color=colors, alpha=0.7)
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{acc:.4f}', ha='center', va='bottom', fontsize=12)
plt.title('Accuracy Comparison of Different CNN Architectures on CIFAR10')
plt.ylabel('Accuracy')
plt.ylim(0, 1.0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 2. Training process comparison
plt.figure(figsize=(14, 5))

# Loss curves
plt.subplot(1, 2, 1)
for name, data in models_data.items():
    plt.plot(data['losses'], color=data['color'], label=name)
plt.title('Training Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)

# Accuracy curves
plt.subplot(1, 2, 2)
for name, data in models_data.items():
    plt.plot(data['accs'], color=data['color'], label=name)
plt.title('Training Accuracy Comparison')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 3. Parameter count calculation
print("\n===== Model Parameter Count Comparison =====\n")
model_instances = {
    'AlexNet': model_alexnet,
    'VGG11': model_vgg,
    'GoogleNet': model_googlenet,
    'ResNet18': model_resnet
}

params_info = {}
for name, model in model_instances.items():
    # 修改1：去掉 .item()，因为 p.numel() 直接返回整数
    total_params = sum(p.numel() for p in model.parameters())
    
    # 修改2：去掉 not，p.requires_grad 为 True 才是可训练参数
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    params_info[name] = total_params
    print(f"{name}: {total_params/1e6:.2f}M parameters (trainable: {trainable_params/1e6:.2f}M)")
    
# 4. Training time comparison
plt.figure(figsize=(10, 6))
times = [models_data[name]['time'] for name in model_names]
bars = plt.bar(model_names, times, color=colors, alpha=0.7)
for bar, t in zip(bars, times):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{t:.1f}s', ha='center', va='bottom', fontsize=12)
plt.title('Training Time Comparison of Different CNN Architectures')
plt.ylabel('Training Time (seconds)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 5. Comprehensive comparison table
print("\n===== Comprehensive Comparison Table =====\n")
print(f"{'Model':<12} {'Accuracy':<10} {'Params':<12} {'Training Time':<12}")
print("-" * 50)
depths = {'AlexNet': '8 layers', 'VGG11': '11 layers', 'GoogleNet': '~14 layers', 'ResNet18': '18 layers'}
for name in model_names:
    acc = models_data[name]['acc']
    params = params_info[name]
    t = models_data[name]['time']
    depth = depths[name]
    print(f"{name:<12} {acc:<8.4f}  {params/1e6:<6.2f}M      {t:<8.1f}s")

print("\nNote: Training time is affected by hardware configuration. The above data is for reference only.")


**Discussion Questions (Model Comparison):**

1. **Accuracy ranking:** How do the 4 networks rank in terms of test accuracy on CIFAR10? Is there a network with significantly higher accuracy than the others? Why?

2. **Parameters vs. accuracy:** Analyze the relationship between parameter count and accuracy. Does the network with the most parameters always have the highest accuracy? Why does GoogleNet perform well despite having relatively few parameters?

3. **Convergence speed:** From the training curves, which network converges fastest? Which is slowest? What do you think are the main factors affecting convergence speed?

4. **Training time:** Which network trains fastest? Which is slowest? Is training time perfectly proportional to parameter count? If training time were not a concern, which network would you choose for a real application? What factors would you consider?

5. **Development trend:** From AlexNet (2012) to ResNet (2015), how did CNN architectures evolve? What pain points did each generation solve compared to its predecessor? Try to summarize each architecture's core contribution in one sentence.


## 9. 9. Summary and Further Discussion

### Summary

In this experiment, we implemented, trained, and evaluated 4 classic CNN architectures on the CIFAR10 dataset:

1. **AlexNet** — The starting point of the deep learning renaissance, demonstrating the effectiveness of deep CNNs, introducing ReLU, Dropout, and data augmentation
2. **VGG** — Demonstrated the effectiveness of stacking small convolutional kernels, with regular structure and strong transferability
3. **GoogleNet (Inception)** — Multi-scale parallel convolutions that improve representational power while controlling computational cost
4. **ResNet** — Residual learning that solves the degradation problem of deep networks, making ultra-deep networks feasible

### Discussion Questions

Please think deeply about the following questions:

**Question 1: Why are CNNs better suited for image tasks than fully connected networks?**

In this comparison, even the worst CNN (AlexNet) achieves much higher accuracy on CIFAR10 than the fully connected networks from previous experiments (~50%). Analyze from the following perspectives:
- Local connectivity vs. global connectivity
- Weight sharing
- Spatial hierarchy (translation invariance)

**Question 2: Analyze the 4 networks from a "feature extractor + classifier" perspective**

All CNNs can be divided into a "feature extractor" and a "classifier". Analyze how each network's feature extractor is designed:
- AlexNet: 5 stacked convolutional layers
- VGG: Grouped convolutions, gradually increasing feature map count
- GoogleNet: Inception modules for parallel multi-scale feature extraction
- ResNet: Stacked residual blocks, features can be transmitted losslessly between layers

**Question 3: Directions of modern CNN development**

This experiment covers classic architectures from 2012-2015. What important directions have CNNs evolved into since then?
- DenseNet: Dense connections
- SENet: Channel attention mechanism
- EfficientNet: Compound scaling
- MobileNet: Lightweight networks
- Transformer (ViT): Completely abandons convolutions

**Question 4: Theory to practice**

Suppose you need to design an image classification system for a mobile app. Which network architecture would you choose? What factors would you consider (accuracy, speed, model size, power consumption, etc.)?

---

*Tip: Compile your answers to the discussion questions into an experiment report and submit it as the written assignment for this experiment.*
